# Неделя 03 — Скрытые пары

Эта базовая модель предсказывает, относится ли скрытая пара к положительному классу (`target = 1`), используя 512 анонимных числовых признаков.

Мы обучаем одно небольшое дерево решений на 2 520 строках обучающей выборки, оцениваем его на неизменённой валидационной выборке из 560 строк и используем тестовую выборку из 1 116 строк только для получения вероятностей. Это учебная базовая модель, а не решение, оптимизированное для лидерборда.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Чтобы использовать Colab, загрузите этот ноутбук вместе с файлами `train.csv`, `validation.csv`, `test.csv` и `sample_submission.csv`, затем запустите все ячейки. После загрузки файлов интернет и GPU не требуются.

## 1. Импорт библиотек и воспроизводимость

pandas используется для работы с CSV-таблицами, NumPy — для числовых проверок, а scikit-learn предоставляет модель и метрику ROC-AUC. PyTorch не используется, поскольку предоставленные файлы уже содержат необходимые 512 числовых признаков.

In [1]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

SEED = 20260916
random.seed(SEED)
np.random.seed(SEED)
notebook_started = time.perf_counter()

## 2. Загрузка предоставленных файлов

Разбиения уже подготовлены. Не создавайте другое случайное разбиение, не объединяйте выборки и не обучайте ничего на тестовых данных.

In [2]:
# DATA_DIR — каталог, содержащий четыре предоставленных CSV-файла.
DATA_DIR = Path(".")

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Обучающая выборка:", train.shape)
print("Валидационная выборка:", validation.shape)
print("Тестовая выборка:", test.shape)
print("Образец отправки:", sample_submission.shape)

Обучающая выборка: (2520, 514)
Валидационная выборка: (560, 514)
Тестовая выборка: (1116, 513)
Образец отправки: (1116, 2)


## 3. Проверка входных данных

Перед моделированием проверьте ожидаемое число строк, обязательные столбцы, пропущенные значения, бинарные целевые значения и одинаковый порядок признаков.

In [3]:
assert train.shape == (2520, 514)
assert validation.shape == (560, 514)
assert test.shape == (1116, 513)
assert sample_submission.shape == (1116, 2)

assert {"row_id", "target"}.issubset(train.columns)
assert {"row_id", "target"}.issubset(validation.columns)
assert "row_id" in test.columns and "target" not in test.columns
assert list(sample_submission.columns) == ["row_id", "target"]

assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()
assert set(train["target"].unique()) == {0, 1}
assert set(validation["target"].unique()) == {0, 1}

## 4. Выбор признаков

`row_id` используется для сопоставления строк, а `target` является ответом. Ни один из этих столбцов не является признаком модели. Во всех выборках должны присутствовать одни и те же 512 упорядоченных столбцов признаков.

### TODO(student) — Задание 1: выбрать столбцы признаков

Создайте `feature_columns` из `train.columns`, исключив `row_id` и `target`. Доступные переменные — это столбцы таблицы `train`. Результат представляет собой упорядоченные входные признаки модели и должен быть списком ровно из 512 названий.

In [4]:
feature_columns = [col for col in train.columns if col not in ["row_id", "target"]]

In [5]:
assert len(feature_columns) == 512
assert not {"row_id", "target"}.intersection(feature_columns)
assert feature_columns == [column for column in validation.columns if column not in {"row_id", "target"}]
assert feature_columns == [column for column in test.columns if column != "row_id"]

## 5. Формирование входных данных модели

Не используйте валидационные данные при обучении. Дерево получает предоставленные числовые столбцы напрямую, поэтому обучаемого шага предобработки нет.

### TODO(student) — Задание 2: сформировать матрицы и целевые векторы

Используя `train`, `validation`, `test` и `feature_columns`, создайте `X_train`, `y_train`, `X_validation`, `y_validation` и `X_test`. Они представляют три матрицы признаков и два доступных целевых вектора. Каждая матрица признаков должна иметь 512 столбцов, а длина каждого целевого вектора должна соответствовать своей размеченной выборке.

In [6]:
X_train = train[feature_columns]
y_train = train["target"]
X_validation = validation[feature_columns]
y_validation = validation["target"]
X_test = test[feature_columns]

In [7]:
assert X_train.shape == (2520, 512)
assert X_validation.shape == (560, 512)
assert X_test.shape == (1116, 512)
assert len(y_train) == len(X_train)
assert len(y_validation) == len(X_validation)

## 6. Определение и обучение базовой модели

Дерево глубины 3 прозрачно и быстро. `min_samples_leaf=20` не позволяет строить правила по очень маленьким группам. Модель обучается только на строках обучающей выборки.

### TODO(student) — Задание 3: создать модель

Создайте `model` как `DecisionTreeClassifier` с параметрами `max_depth=3`, `min_samples_leaf=20` и `random_state=SEED`. Результатом должен быть ещё не обученный классификатор.

In [8]:
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED)

### TODO(student) — Задание 4: обучить модель

Обучите `model`, используя только `X_train` и `y_train`. Результатом должен быть классификатор, обученный исключительно на обучающих строках и имеющий метод `predict_proba`.

In [9]:
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",20260916
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the cur

## 7. Оценка на валидационной выборке

ROC-AUC показывает, насколько хорошо непрерывные оценки ранжируют положительные примеры выше отрицательных при всех возможных порогах. Классовые метки после применения одного порога теряют информацию о ранжировании, поэтому их не следует использовать для расчёта ROC-AUC.

### TODO(student) — Задание 5: валидационные вероятности и ROC-AUC

Используя `model`, `X_validation` и `y_validation`, создайте вероятности класса 1 `validation_probability` и скаляр `validation_auc`. На каждую валидационную строку должна приходиться одна вероятность, а AUC должен быть конечным и приблизительно равным 0.7473.

In [10]:
validation_probability = model.predict_proba(X_validation)[:, 1]
validation_auc = roc_auc_score(y_validation, validation_probability)

In [11]:
assert validation_probability.shape == (len(validation),)
assert np.isfinite(validation_auc)
assert np.isclose(validation_auc, 0.7473086734693878)
assert 0.0 <= validation_auc <= 1.0

## 8. Предсказание для тестовой выборки

Тестовая выборка остаётся неразмеченной и используется только после обучения. Повторное предсказание проверяет детерминированность. Ориентировочный публичный результат преподавателя равен примерно 0.72157; публичные метки недоступны этому ноутбуку и не используются в нём.

### TODO(student) — Задание 6: вероятности для тестовой выборки

Используя `model` и `X_test`, создайте вероятности класса 1 `test_probability`. Повторите предсказание и сохраните его как `test_probability_repeat`; оба массива должны быть одинаковыми и содержать по одной вероятности из диапазона `[0, 1]` для каждой тестовой строки.

In [12]:
test_probability = model.predict_proba(X_test)[:, 1]
test_probability_repeat = model.predict_proba(X_test)[:, 1]

## 9. Создание и проверка файла отправки

Использование образца отправки сохраняет требуемый порядок строк и схему. Замените только столбец `target` вероятностями положительного класса.

### TODO(student) — Задание 7: заполнить и сохранить файл отправки

Скопируйте `sample_submission` в `submission`, замените целевые значения на `test_probability`, проверьте схему из двух столбцов, число строк, точный порядок тестовых ID, уникальность ID, отсутствие пропусков и диапазон вероятностей, затем сохраните `submission.csv` с `index=False`.

In [13]:
submission = sample_submission.copy()
submission["target"] = test_probability

assert submission.shape == (len(test), 2)
assert submission["row_id"].equals(test["row_id"])
assert submission["row_id"].is_unique
assert not submission.isna().sum().any()
assert all(0 <= submission["target"][i] <= 1 for i in range(len(submission)))

OUTPUT_DIR = DATA_DIR / "runs"
OUTPUT_DIR.mkdir(exist_ok=True)
submission.to_csv(OUTPUT_DIR / "submission_tree.csv", index=False)

In [14]:
runtime_секунд = time.perf_counter() - notebook_started
print(f"ROC-AUC на валидации: {validation_auc:.6f}")
print(f"Время работы: {runtime_секунд:.2f} секунд")
print(submission.head(100))

ROC-AUC на валидации: 0.747309
Время работы: 1.80 секунд
                      row_id    target
0   row_46ad1afad8562239f79c  0.865625
1   row_b42ade9cd57b8d919acb  0.865625
2   row_c951cd5e197b6b4e8b83  0.611111
3   row_a54b3423760d599416ce  0.322105
4   row_a87c87cc47690e2a303f  0.322105
..                       ...       ...
95  row_d27a77d631c27b3f4947  0.084249
96  row_01b0d2ff21b4ba678b4d  0.286245
97  row_14c523ab600715fb1a7d  0.577348
98  row_8037c9a1d7d1a3fea833  0.340278
99  row_416e89831d1c39a2de82  0.865625

[100 rows x 2 columns]


## TODO(student) — Задание 8: самостоятельное сравнение моделей

Обучите и сравните дерево решений, логистическую регрессию, случайный лес и градиентный бустинг, используя предоставленное разбиение train/validation. Представьте ROC-AUC на валидации и время работы каждой модели в компактной таблице, затем обоснуйте итоговый выбор. Не обучайте и не настраивайте модели на валидационных данных.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [16]:
log_reg = LogisticRegression(max_iter=2000, random_state=SEED)
rf = RandomForestClassifier(max_depth=5, min_samples_leaf=20, n_estimators=200, random_state=SEED)
gb = GradientBoostingClassifier(n_estimators=100, random_state=SEED)

In [17]:
models = {
    "Decision Tree": model,
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

results = []
for name, current_model in models.items():
    start_time = time.perf_counter()
    current_model.fit(X_train, y_train)
    probability = current_model.predict_proba(X_validation)[:, 1]
    auc = roc_auc_score(y_validation, probability)
    elapsed_time = time.perf_counter() - start_time
    results.append({
        "Модель": name,
        "ROC-AUC": auc,
        "Время, сек": elapsed_time
    })

results_df = pd.DataFrame(results)
results_df

,Модель,ROC-AUC,"Время, сек"
0,Decision Tree,0.747309,0.787953
1,Logistic Regression,0.890293,0.307933
2,Random Forest,0.884490,6.442394
3,Gradient Boosting,0.905306,63.288467
